# Panel de balance mensual — pre-2020

Panel auxiliar con saldos mensuales en **moneda corriente** desde noviembre 1994 hasta diciembre 2019. Se usa para:

1. Ejercicio de robustez con el blanqueo 2016-17 (Ley 27.260, gobierno Macri) como contrafactual descriptivo.
2. Estudios de series largas que excedan la ventana principal del paper.

Formato idéntico al panel principal, pero con una flag adicional `moneda_homogenea = False` que recuerda que los saldos no son directamente comparables con los del panel principal sin un ajuste.

Decisión metodológica (§3.3): los dos paneles se mantienen separados. La unificación de saldos pre y post 2020 es un problema abierto que involucra NIIF 9 y NIC 29; no se resuelve en esta etapa.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "utils"))
from paths import RAW, INTERIM, DIMENSIONES, PANELES, REPO

import pandas as pd
import duckdb

BALHIST = RAW / "bcra_ief/202601/Entfin/Tec_Cont/bal_hist/balhist.txt"
OUT = PANELES / "panel_balance_mensual_pre2020.parquet"
OUT.parent.mkdir(parents=True, exist_ok=True)

FECHA_LIMITE = 202001  # yyyymm exclusive

## Lectura

In [ ]:
raw = pd.read_csv(
    BALHIST,
    sep="\t",
    header=None,
    names=["codigo_entidad", "yyyymm", "codigo_cuenta", "saldo_miles_pesos"],
    encoding="ISO-8859-1",
    dtype={"codigo_entidad": str, "yyyymm": str, "codigo_cuenta": str, "saldo_miles_pesos": float},
)
print(f"Filas leídas: {len(raw):,}")

## Filtros

In [ ]:
raw["yyyymm_int"] = raw["yyyymm"].astype(int)
panel = raw[raw["yyyymm_int"] < FECHA_LIMITE].copy()
print(f"Post-filtro temporal (< {FECHA_LIMITE}): {len(panel):,} filas")

panel = panel[~panel["codigo_entidad"].str.startswith("AA")].copy()
print(f"Post-filtro entidades: {len(panel):,} filas")

## Columnas derivadas

In [ ]:
panel["fecha"] = pd.to_datetime(panel["yyyymm"], format="%Y%m") + pd.offsets.MonthEnd(0)
panel["saldo"] = panel["saldo_miles_pesos"] * 1000
panel["moneda_homogenea"] = False

columnas = ["codigo_entidad", "yyyymm", "fecha", "codigo_cuenta", "saldo", "moneda_homogenea"]
panel = panel[columnas].copy()
panel["yyyymm"] = panel["yyyymm"].astype(int)

## Persistencia

In [ ]:
panel.to_parquet(OUT, index=False)
print(f"Escrito: {OUT.relative_to(REPO)}")
print(f"Filas: {len(panel):,}")

## Validaciones

In [ ]:
assert panel["yyyymm"].max() < FECHA_LIMITE
assert panel["yyyymm"].min() >= 199411, "Se esperaba cobertura desde 1994-11"
assert not panel["codigo_entidad"].str.startswith("AA").any()
assert (~panel["moneda_homogenea"]).all()

resumen = {
    "filas": len(panel),
    "entidades": panel["codigo_entidad"].nunique(),
    "cuentas": panel["codigo_cuenta"].nunique(),
    "meses": panel["yyyymm"].nunique(),
    "fecha_min": panel["yyyymm"].min(),
    "fecha_max": panel["yyyymm"].max(),
}
print("Validaciones OK")
for k, v in resumen.items():
    print(f"  {k}: {v:,}")

## Chequeo del blanqueo 2016-17

Búsqueda de las cuentas especiales de la Ley 27.260 (blanqueo Macri) en el panel pre-2020.

In [ ]:
duckdb.sql(f"""
    select p.yyyymm, count(distinct p.codigo_entidad) as entidades_con_saldo,
           sum(p.saldo) / 1e9 as saldo_total_miles_millones
    from '{OUT}' p
    where p.codigo_cuenta in ('311781', '311782', '311783', '315781', '315782', '315783')
      and p.saldo > 0
    group by p.yyyymm
    order by p.yyyymm
""").df()